# MIND Kaggle runner

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!git clone --branch main --depth 1 https://github.com/vohuutridung/IT4613.git /kaggle/working/IT4613
%cd /kaggle/working/IT4613

In [ ]:
# # Preprocess raw MIND once. Save /kaggle/working/mind15/prepared as a Kaggle Dataset afterward.
# !bash deployment/kaggle/run.sh \
#   --mode prepare \
#   --data-root /kaggle/input/datasets/vhutrdng/microsoft-news-dataset \
#   --work-root /kaggle/working/mind15

# # Zip folder /kaggle/working/mind15/prepared for downloading
# !cd /kaggle/working/mind15 && zip -r prepared.zip prepared/

In [ ]:
# SINGLE MODEL: train partition 0 and create submission.zip.
# Replace prepared-root with /kaggle/input/<dataset>/prepared when reusing a saved bundle.
!bash deployment/kaggle/run.sh \
  --mode single \
  --prepared-root /kaggle/input/datasets/vhutrdng/microsoft-news-dataset/prepared_v1 \
  --partition 0 \
  --effective-batch 1024 \
  --batch-size 64 \
  --replicas 2 \
  --precision mixed_float16 \
  --work-root /kaggle/working/mind15

In [ ]:
# ENSEMBLE (optional): train partitions 0..4 sequentially.
!bash deployment/kaggle/run.sh \
  --mode train-ensemble \
  --effective-batch 1024 \
  --batch-size 64 \
  --replicas 2 \
  --prepared-root /kaggle/input/CHANGE_ME_PREPARED/prepared \
  --work-root /kaggle/working/mind15

In [ ]:
# ENSEMBLE SUBMISSION (optional): mean probabilities from partitions 0..4.
!bash deployment/kaggle/run.sh \
  --mode ensemble-submit \
  --prepared-root /kaggle/input/CHANGE_ME_PREPARED/prepared \
  --work-root /kaggle/working/mind15

In [ ]:
# OPTIONAL: upload one completed run to Hugging Face.
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

!bash deployment/kaggle/run.sh \
  --mode push-model \
  --run-dir /kaggle/working/mind15/recommendation_runs/partition-0 \
  --hf-folder experiments/partition-0 \
  --work-root /kaggle/working/mind15